# Brain Tumor Detection and Classification using DenseNet121

An end-to-end academic machine learning and computer vision pipeline for four-class brain MRI classification using transfer learning with DenseNet121.

> [!CAUTION]
> **Medical Disclaimer:** This project is developed strictly for academic and research purposes. It is not a certified diagnostic system, does not medically localize tumors, and must never be used to replace professional medical diagnosis by certified radiologists or oncologists.

---

## 1. Problem Statement
Brain tumors represent one of the most critical central nervous system pathologies. Timely classification of tumor histology from Magnetic Resonance Imaging (MRI) is essential for treatment planning. 

In this project, we implement a four-class classification pipeline to distinguish:
1. **Glioma** (`glioma`): Tumors arising from glial tissue in the brain.
2. **Meningioma** (`meningioma`): Typically benign tumors developing from the membranous layers surrounding the brain.
3. **No Tumor** (`notumor`): Normal brain MRI scans without detectable neoplastic lesions.
4. **Pituitary Tumor** (`pituitary`): Neoplasms arising from the pituitary gland at the base of the brain.

The principal deep-learning architecture is ImageNet-pretrained **DenseNet121**, adapted through a two-stage transfer learning and fine-tuning strategy.


## 2. Environment Diagnostics & Library Imports
We verify the runtime environment, Python version, TensorFlow release, NumPy release, and GPU availability before starting pipeline execution.


In [ ]:
import sys
import platform
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

print(f"Python version     : {platform.python_version()} ({platform.system()})")
print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")

# GPU device detection
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU detected       : True ({len(gpus)} device(s): {[g.name for g in gpus]})")
else:
    print("GPU detected       : False (GPU not detected. Training will use CPU and may take longer.)")

try:
    from IPython.display import display
except ImportError:
    display = print


## 3. Project Root & Path Discovery
To ensure the notebook executes reliably whether launched from the repository root, `notebooks/`, VS Code, JupyterLab, or command line, we dynamically discover the repository root using project markers (`src`, `notebooks`, `README.md`).


In [ ]:
# Dynamically resolve project root
current_path = Path.cwd().resolve()
candidates = [current_path, *current_path.parents]
project_root = None

for candidate in candidates:
    if (candidate / "src").is_dir() and ((candidate / "README.md").is_file() or (candidate / "notebooks").is_dir()):
        project_root = candidate
        break

if project_root is None:
    project_root = current_path if current_path.name != "notebooks" else current_path.parent

PROJECT_ROOT = project_root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Resolved Project Root:", PROJECT_ROOT)

# Import modular project utilities
import src.config as config
from src.dataset_utils import (
    find_dataset_root,
    inspect_dataset_structure,
    validate_dataset_structure,
    DatasetValidationError,
    CANONICAL_CLASSES,
)
from src.data_loader import load_datasets
from src.eda import analyze_dataset, run_eda
from src.preprocessing import load_rgb_image, image_to_model_batch, create_data_augmentation
from src.model import build_densenet121_model, find_backbone, prepare_for_fine_tuning
from src.train import set_reproducible_seeds
from src.predict import predict_with_model, load_trained_model
from app.gradcam import create_heatmap_images, find_last_conv_layer

set_reproducible_seeds(config.RANDOM_SEED)
print("[OK] Core project modules imported successfully.")


## 4. Project and Dataset Configuration
Configure dataset location and pipeline hyperparameters. The system supports custom dataset paths and the `BRAIN_TUMOR_DATASET_DIR` environment variable.


In [ ]:
# =============================================================================
# DATASET PATH CONFIGURATION
# Default: points to '<project_root>/dataset'
# If your dataset is stored elsewhere, uncomment and customize the path below:
# DATASET_PATH = Path("D:/Datasets/Brain Tumor MRI Dataset")
# =============================================================================
DATASET_PATH = PROJECT_ROOT / "dataset"

# Display pipeline hyperparameter table
config_summary = pd.DataFrame({
    "Hyperparameter / Setting": [
        "Project Root", "Dataset Root", "Target Image Size", "Input Tensor Shape",
        "Number of Classes", "Canonical Class Order", "Batch Size",
        "Train / Validation Split", "Initial Epochs (Head)", "Fine-Tune Epochs",
        "Initial Learning Rate", "Fine-Tune Learning Rate", "Fine-Tune Layers",
        "Random Seed"
    ],
    "Configured Value": [
        str(PROJECT_ROOT), str(DATASET_PATH), str(config.IMAGE_SIZE),
        str(config.INPUT_SHAPE), config.NUM_CLASSES,
        ", ".join(config.CLASS_NAMES), config.BATCH_SIZE,
        f"{int((1 - config.VALIDATION_SPLIT)*100)}% / {int(config.VALIDATION_SPLIT*100)}%",
        config.INITIAL_EPOCHS, config.FINE_TUNE_EPOCHS,
        f"{config.INITIAL_LEARNING_RATE:.1e}", f"{config.FINE_TUNE_LEARNING_RATE:.1e}",
        config.FINE_TUNE_LAYERS, config.RANDOM_SEED
    ]
})
display(config_summary)


## 5. Dataset Setup and Validation (Diagnostic Cell)
Before initializing TensorFlow data pipelines, we inspect the dataset root, verify directory structures, count available images per class, detect file extensions, and check for missing folders or corrupt files.


In [ ]:
# Comprehensive Dataset Diagnostic Scan
diagnostic_info = inspect_dataset_structure(dataset_dir=DATASET_PATH, project_root=PROJECT_ROOT)

print(f"Dataset root       : {diagnostic_info['dataset_root']}")
print(f"Training directory : {diagnostic_info['training_dir']}")
print(f"Testing directory  : {diagnostic_info['testing_dir']}")
print(f"Discovered folders : {diagnostic_info['discovered_folders']}")
print("\nDetected classes and counts:")

print("Training Split:")
for canonical in CANONICAL_CLASSES:
    c_info = diagnostic_info['training_classes'].get(canonical, {})
    status = f"{c_info.get('image_count', 0):,d} images (folder: '{c_info.get('folder_name', 'MISSING')}')"
    print(f"  {config.DISPLAY_NAMES[canonical]:18s}: {status}")

print("\nTesting Split:")
for canonical in CANONICAL_CLASSES:
    c_info = diagnostic_info['testing_classes'].get(canonical, {})
    status = f"{c_info.get('image_count', 0):,d} images (folder: '{c_info.get('folder_name', 'MISSING')}')"
    print(f"  {config.DISPLAY_NAMES[canonical]:18s}: {status}")

if diagnostic_info["is_valid"]:
    print("\n[OK] Dataset validation successful. Ready for training and evaluation.")
else:
    print("\n[ERROR] Dataset validation failed!")
    print(diagnostic_info["error_message"])


## 6. Exploratory Data Analysis (EDA)
EDA is performed directly on physical files on disk. We compute the class distribution, image dimensions across classes, corrupted files, and class imbalance ratio.


In [ ]:
eda_stats = run_eda(training_dir=diagnostic_info['training_dir'])

eda_table = pd.DataFrame({
    "EDA Metric": [
        "Total Training Images", "Number of Classes",
        "Max-to-Min Imbalance Ratio", "Corrupted / Unreadable Files"
    ],
    "Observed Value": [
        f"{eda_stats['total_images']:,d}",
        eda_stats['number_of_classes'],
        f"{eda_stats['potential_imbalance_ratio_max_to_min']:.2f}" if eda_stats['potential_imbalance_ratio_max_to_min'] else "1.00",
        len(eda_stats['corrupted_files'])
    ]
})
display(eda_table)

if eda_stats['image_dimensions']:
    dim_df = pd.DataFrame(
        list(eda_stats['image_dimensions'].items())[:10],
        columns=["Original Image Dimension", "Frequency"]
    )
    print("Most frequent original image dimensions:")
    display(dim_df)


## 7. Class Distribution Visualization
Visualizing the distribution of training samples across the four tumor categories.


In [ ]:
counts = eda_stats["images_per_class"]
labels = [config.DISPLAY_NAMES[name] for name in config.CLASS_NAMES]
values = [counts[name] for name in config.CLASS_NAMES]
colors = ["#2b6cb0", "#319795", "#d69e2e", "#805ad5"]

plt.figure(figsize=(9, 4.5))
bars = plt.bar(labels, values, color=colors, edgecolor="black", linewidth=1.2, width=0.55)
plt.title("Training Set Class Distribution", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Brain MRI Category", fontsize=11, labelpad=8)
plt.ylabel("Number of Slices", fontsize=11, labelpad=8)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, height + 20, f"{height:,d}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.ylim(0, max(values) * 1.15)
plt.tight_layout()
plt.show()


## 8. Sample MRI Visualization
Inspecting real brain MRI slices from each class to observe morphological patterns.


In [ ]:
sample_images = []
for class_name in config.CLASS_NAMES:
    class_folder = diagnostic_info["training_classes"][class_name]["path"]
    if class_folder and class_folder.is_dir():
        image_files = sorted([p for p in class_folder.iterdir() if p.suffix.lower() in config.SUPPORTED_EXTENSIONS])
        if image_files:
            sample_images.append((class_name, image_files[0]))

fig, axes = plt.subplots(1, len(sample_images), figsize=(16, 4))
if len(sample_images) == 1:
    axes = [axes]

for ax, (cls_name, img_path) in zip(axes, sample_images):
    with Image.open(img_path) as img:
        ax.imshow(img.convert("RGB"))
    ax.set_title(f"{config.DISPLAY_NAMES[cls_name]}\n({img_path.name})", fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("Representative Sample MRI Slices per Category", fontsize=13, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()


## 9. Image Preprocessing Pipeline
Pretrained convolutional networks like DenseNet121 require specific input standardization:
1. **Readable Image Validation**: Check that image bytes are non-corrupted and readable.
2. **RGB Conversion**: Convert grayscale MRI slices to 3-channel RGB (`(224, 224, 3)`).
3. **Resizing**: Standardize to $224 \times 224$ pixels using Lanczos antialiasing.
4. **Float32 Casting**: Cast pixel values from `uint8` [0, 255] to `float32`.
5. **DenseNet Normalization**: Apply `tf.keras.applications.densenet.preprocess_input`, which scales pixels to $[-1, 1]$ based on ImageNet normalization.


In [ ]:
demo_img_path = sample_images[0][1]

with Image.open(demo_img_path) as raw_img:
    raw_w, raw_h = raw_img.size
    raw_mode = raw_img.mode
    raw_np = np.asarray(raw_img)

processed_rgb = load_rgb_image(demo_img_path)
model_tensor = image_to_model_batch(demo_img_path)

print(f"Raw image file          : {demo_img_path.name}")
print(f"Original mode & size    : {raw_mode} ({raw_w} x {raw_h})")
print(f"Raw pixel range         : [{raw_np.min()}, {raw_np.max()}] (dtype={raw_np.dtype})")
print(f"Processed PIL mode/size : {processed_rgb.mode} {processed_rgb.size}")
print(f"Model batch shape       : {model_tensor.shape} (dtype={model_tensor.dtype})")
print(f"Normalized tensor range : [{model_tensor.min():.3f}, {model_tensor.max():.3f}]")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(raw_img, cmap="gray" if raw_mode == "L" else None)
axes[0].set_title(f"Original MRI ({raw_w}x{raw_h})", fontsize=11, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(processed_rgb)
axes[1].set_title("RGB Standardized (224x224)", fontsize=11, fontweight="bold")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 10. Data Augmentation
To prevent overfitting and enhance generalization on unseen MRI orientations, conservative augmentations are applied **only** to training slices:
- **RandomRotation** ($\pm 4\%$)
- **RandomZoom** ($\pm 8\%$)
- **RandomTranslation** ($\pm 4\%$)
- **RandomContrast** ($\pm 8\%$)

Extreme transformations (such as vertical flips or strong shearing) are deliberately avoided because brain MRI orientations encode anatomical priors.


In [ ]:
augmentation_pipeline = create_data_augmentation(seed=config.RANDOM_SEED)

sample_tensor = tf.cast(np.asarray(processed_rgb), tf.float32)
augmented_samples = [
    augmentation_pipeline(tf.expand_dims(sample_tensor, 0), training=True)[0].numpy()
    for _ in range(6)
]

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, aug_img, idx in zip(axes.ravel(), augmented_samples, range(1, 7)):
    ax.imshow(np.clip(aug_img / 255.0, 0.0, 1.0))
    ax.set_title(f"Augmented Variation #{idx}", fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("Conservative Training Augmentation Variations", fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()


## 11. Prepare Training, Validation, and Test Data
We create memory-efficient `tf.data.Dataset` streams using prefetching (`tf.data.AUTOTUNE`).
- **Training Set (80%)**: Sourced from `Training/` directory, augmented, preprocessed.
- **Validation Set (20%)**: Sourced from `Training/` directory, unaugmented, preprocessed.
- **Test Set (100%)**: Sourced from held-out `Testing/` directory, unaugmented, preprocessed.


In [ ]:
BATCH_SIZE = config.BATCH_SIZE

train_ds, val_ds, test_ds = load_datasets(
    dataset_dir=DATASET_PATH,
    batch_size=BATCH_SIZE,
    validation_split=config.VALIDATION_SPLIT,
    random_seed=config.RANDOM_SEED
)

train_batch_img, train_batch_lbl = next(iter(train_ds))
val_batch_img, val_batch_lbl = next(iter(val_ds))
test_batch_img, test_batch_lbl = next(iter(test_ds))

print("TensorFlow Dataset Pipeline Initialized Successfully:")
print(f"Training batch   : images {train_batch_img.shape}, labels {train_batch_lbl.shape}")
print(f"Validation batch : images {val_batch_img.shape}, labels {val_batch_lbl.shape}")
print(f"Testing batch    : images {test_batch_img.shape}, labels {test_batch_lbl.shape}")
print(f"Class names      : {config.CLASS_NAMES}")


## 12. Build DenseNet121 Architecture
We instantiate the pretrained DenseNet121 backbone without its original 1000-class ImageNet top, freeze all backbone parameters, and append a custom classification head designed for four-class MRI classification.


In [ ]:
model, backbone = build_densenet121_model(weights="imagenet")

total_params = model.count_params()
trainable_params = sum(np.prod(v.shape) for v in model.trainable_weights)
non_trainable_params = total_params - trainable_params

print("DenseNet121 Model Built:")
print(f"Model Name           : {model.name}")
print(f"Backbone Name        : {backbone.name}")
print(f"Backbone Frozen      : {not backbone.trainable}")
print(f"Total Parameters     : {total_params:,d}")
print(f"Trainable Parameters : {int(trainable_params):,d} (Classification Head)")
print(f"Non-Trainable Params : {int(non_trainable_params):,d} (Frozen DenseNet Backbone)")

model.summary(line_length=100)


## 13. Explain DenseNet121 Architecture
### Mathematical Intuition
In a standard Convolutional Neural Network (CNN) with $L$ layers, there are $L$ connections—one between each layer and its subsequent layer:
$$x_l = H_l(x_{l-1})$$

In **DenseNet** (Huang et al., 2017), each layer obtains additional inputs from all preceding layers and passes on its own feature maps to all subsequent layers:
$$x_l = H_l([x_0, x_1, x_2, \dots, x_{l-1}])$$
where $[x_0, x_1, \dots, x_{l-1}]$ represents the concatenation of all feature maps produced in layers $0, 1, \dots, l-1$.

### Key Advantages for Brain MRI Classification:
1. **Feature Reuse:** Low-level edge and texture representations learned in shallow layers are preserved throughout the network rather than being lost through successive convolutions.
2. **Gradient Highway:** Because each layer is connected to the loss function through direct connections, vanishing gradients are mitigated during backpropagation.
3. **Parameter Efficiency:** Despite its depth (121 layers), DenseNet121 requires substantially fewer parameters (~7 million) than ResNet50 (~25 million) or VGG16 (~138 million), making it well-suited for academic hardware.


## 14. Stage 1 — Transfer Learning Setup & Execution Controls
In Stage 1 (Feature Extraction), the DenseNet121 backbone is completely frozen. Only the new classification head (GlobalAveragePooling2D, BatchNormalization, Dense(256), Dropout(0.35), Softmax(4)) is trained using Adam ($lr = 1 \times 10^{-3}$).

> **Execution Controls:** Set `RUN_TRAINING = True` if you wish to run model training from the notebook. Use `QUICK_TEST = True` to run 1 epoch per stage for quick pipeline verification.


In [ ]:
# =============================================================================
# NOTEBOOK TRAINING EXECUTION CONTROLS
# Set RUN_TRAINING = True to execute training directly from this notebook.
# Set QUICK_TEST = True for a rapid 1-epoch verification run.
# Set QUICK_TEST = False for full academic training (12 Stage-1 + 8 Stage-2 epochs).
# =============================================================================
RUN_TRAINING = False
QUICK_TEST = False

stage1_epochs = 1 if QUICK_TEST else config.INITIAL_EPOCHS
stage2_epochs = 1 if QUICK_TEST else config.FINE_TUNE_EPOCHS

print(f"RUN_TRAINING  : {RUN_TRAINING}")
print(f"QUICK_TEST    : {QUICK_TEST}")
print(f"Stage 1 Epochs: {stage1_epochs}")
print(f"Stage 2 Epochs: {stage2_epochs}")
print(f"Trained Model Exists: {config.MODEL_PATH.exists()}")


## 15. Train Classification Head (Stage 1 Feature Extraction)
Training the classification head while the convolutional backbone remains frozen.


In [ ]:
from src.train import make_callbacks, merge_histories, _history_to_dict

if RUN_TRAINING:
    print(f"Starting Stage 1: Feature Extraction ({stage1_epochs} epochs)...")
    stage1_callbacks = make_callbacks(config.STAGE1_MODEL_PATH)
    stage1_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=stage1_epochs,
        callbacks=stage1_callbacks,
        verbose=1
    )
    print("[OK] Stage 1 training completed.")
else:
    print("RUN_TRAINING is False. Skipping Stage 1 training.")
    if config.STAGE1_MODEL_PATH.exists():
        print(f"Existing Stage 1 checkpoint found: {config.STAGE1_MODEL_PATH}")
    else:
        print("No Stage 1 checkpoint exists yet. Set RUN_TRAINING = True to train.")


## 16. Stage 1 Training Curves
Plotting training versus validation accuracy and loss during the feature extraction stage.


In [ ]:
if RUN_TRAINING and 'stage1_history' in locals():
    h1 = stage1_history.history
    epochs_range = range(1, len(h1['loss']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(epochs_range, h1['accuracy'], label='Train Accuracy', marker='o')
    axes[0].plot(epochs_range, h1['val_accuracy'], label='Val Accuracy', marker='s')
    axes[0].set_title('Stage 1 Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs_range, h1['loss'], label='Train Loss', marker='o', color='orange')
    axes[1].plot(epochs_range, h1['val_loss'], label='Val Loss', marker='s', color='red')
    axes[1].set_title('Stage 1 Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Stage 1 training history pending actual model run.")


## 17. Stage 2 — Selective Fine-Tuning Setup
In Stage 2, we:
1. Reload the best weights saved from Stage 1 (`models/stage1_best.keras`).
2. Unfreeze only the upper 40 layers of DenseNet121, allowing high-level filters to adapt to MRI-specific features.
3. Keep all `BatchNormalization` layers frozen to preserve running mean and variance estimates.
4. Reduce the learning rate by a factor of 100 ($lr = 1 \times 10^{-5}$) to prevent destabilizing pretrained weights.


In [ ]:
print("Configuring Selective Fine-Tuning:")
print(f"Upper layers to unfreeze : {config.FINE_TUNE_LAYERS}")
print(f"Fine-tune learning rate  : {config.FINE_TUNE_LEARNING_RATE} (vs initial {config.INITIAL_LEARNING_RATE})")
print(f"BatchNormalization frozen: True")


## 18. Fine-Tuning Execution & Merged Curves
Executing Stage 2 fine-tuning and generating merged learning curves showing the transition from feature extraction to fine-tuning.


In [ ]:
if RUN_TRAINING:
    print(f"Reloading best Stage 1 model from {config.STAGE1_MODEL_PATH}...")
    stage1_best = tf.keras.models.load_model(config.STAGE1_MODEL_PATH)
    fine_tune_model = prepare_for_fine_tuning(stage1_best, config.FINE_TUNE_LAYERS)
    
    print(f"Starting Stage 2: Selective Fine-Tuning ({stage2_epochs} epochs)...")
    stage2_callbacks = make_callbacks(config.STAGE2_MODEL_PATH)
    stage2_history = fine_tune_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=stage2_epochs,
        callbacks=stage2_callbacks,
        verbose=1
    )
    
    merged_history = merge_histories(
        _history_to_dict(stage1_history),
        _history_to_dict(stage2_history)
    )
    config.TRAINING_HISTORY_PATH.write_text(json.dumps(merged_history, indent=2), encoding="utf-8")
    print("[OK] Stage 2 fine-tuning completed. History saved to", config.TRAINING_HISTORY_PATH)
else:
    print("RUN_TRAINING is False. Skipping Stage 2 fine-tuning.")
    if config.STAGE2_MODEL_PATH.exists():
        print(f"Existing Stage 2 checkpoint found: {config.STAGE2_MODEL_PATH}")
    else:
        print("No Stage 2 checkpoint exists yet. Set RUN_TRAINING = True to train.")


## 19. Final Model Selection and Checkpoint Saving
We evaluate checkpoints from both Stage 1 and Stage 2 against the validation set and save the overall best model to `models/best_densenet121.keras`.


In [ ]:
if RUN_TRAINING:
    candidates = []
    for name, path in [("Stage 1 (Head)", config.STAGE1_MODEL_PATH), ("Stage 2 (Fine-Tuning)", config.STAGE2_MODEL_PATH)]:
        if path.exists():
            candidate_m = tf.keras.models.load_model(path)
            loss, acc = candidate_m.evaluate(val_ds, verbose=0)
            candidates.append((candidate_m, name, float(loss), float(acc)))
            print(f"Checkpoint '{name}': Validation Loss = {loss:.4f}, Accuracy = {acc*100:.2f}%")
            
    if candidates:
        best_candidate = min(candidates, key=lambda c: c[2])
        best_candidate[0].save(config.MODEL_PATH)
        print(f"\n[OK] Saved overall best model ({best_candidate[1]}) to: {config.MODEL_PATH}")
elif config.MODEL_PATH.exists():
    print(f"[OK] Using existing trained model: {config.MODEL_PATH}")
else:
    print("No trained model available yet. Set RUN_TRAINING = True to train.")


## 20. Evaluate Test Dataset
We evaluate the final saved model on the held-out test dataset (`Testing/`), which was never seen during Stage 1 training, Stage 2 fine-tuning, or hyperparameter validation.


In [ ]:
if config.MODEL_PATH.exists():
    print(f"Loading final trained model from: {config.MODEL_PATH}")
    eval_model = tf.keras.models.load_model(config.MODEL_PATH)
    
    print("Computing predictions on test split...")
    test_probabilities = eval_model.predict(test_ds, verbose=1)
    test_y_true = np.concatenate([np.argmax(lbl.numpy(), axis=1) for _, lbl in test_ds])
    test_y_pred = np.argmax(test_probabilities, axis=1)
    print(f"[OK] Computed predictions for {len(test_y_true)} test images.")
else:
    print("No trained model found at models/best_densenet121.keras.")
    print("Evaluation will execute automatically once training is completed.")


## 21. Accuracy, Precision, Recall, and F1-Score
Computing macroscopic summary classification metrics from the actual test predictions.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

if 'test_y_true' in locals():
    overall_acc = accuracy_score(test_y_true, test_y_pred)
    macro_prec = precision_score(test_y_true, test_y_pred, average="macro", zero_division=0)
    macro_rec = recall_score(test_y_true, test_y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(test_y_true, test_y_pred, average="macro", zero_division=0)
    
    test_metrics_df = pd.DataFrame({
        "Metric": ["Test Accuracy", "Macro Precision", "Macro Recall", "Macro F1-Score", "Test Samples"],
        "Value": [f"{overall_acc*100:.2f}%", f"{macro_prec*100:.2f}%", f"{macro_rec*100:.2f}%", f"{macro_f1*100:.2f}%", str(len(test_y_true))]
    })
    display(test_metrics_df)
else:
    print("Metrics pending actual model training.")


## 22. Classification Report
Detailed class-wise precision, recall, and F1-score across Glioma, Meningioma, No Tumor, and Pituitary categories.


In [ ]:
from sklearn.metrics import classification_report

if 'test_y_true' in locals():
    report_dict = classification_report(
        test_y_true,
        test_y_pred,
        labels=list(range(len(config.CLASS_NAMES))),
        target_names=list(config.CLASS_NAMES),
        output_dict=True,
        zero_division=0
    )
    
    class_rows = []
    for cls_name in config.CLASS_NAMES:
        cls_metrics = report_dict[cls_name]
        class_rows.append({
            "Tumor Category": config.DISPLAY_NAMES[cls_name],
            "Precision": f"{cls_metrics['precision']:.4f}",
            "Recall": f"{cls_metrics['recall']:.4f}",
            "F1-Score": f"{cls_metrics['f1-score']:.4f}",
            "Support (Images)": int(cls_metrics['support'])
        })
    display(pd.DataFrame(class_rows))
else:
    print("Classification report pending actual model training.")


## 23. Confusion Matrix
Plotting the confusion matrix showing true labels versus model predictions.


In [ ]:
from sklearn.metrics import confusion_matrix

if 'test_y_true' in locals():
    cm = confusion_matrix(test_y_true, test_y_pred, labels=list(range(len(config.CLASS_NAMES))))
    labels = [config.DISPLAY_NAMES[name] for name in config.CLASS_NAMES]
    
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=ax)
    
    ax.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        ylabel='True Diagnostic Category',
        xlabel='Model Predicted Category',
        title='DenseNet121 Brain MRI Confusion Matrix'
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    
    thresh = cm.max() / 2.0 if cm.max() > 0 else 1.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontweight="bold")
                    
    plt.tight_layout()
    plt.savefig(config.PLOTS_DIR / "confusion_matrix.png", dpi=160)
    plt.show()
else:
    print("Confusion matrix pending actual model training.")


## 24. One-vs-Rest ROC / AUC Analysis
Computing Receiver Operating Characteristic (ROC) curves and Area Under the Curve (AUC) for multiclass evaluation.


In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

if 'test_y_true' in locals():
    y_true_bin = label_binarize(test_y_true, classes=np.arange(len(config.CLASS_NAMES)))
    auc_records = {}
    
    plt.figure(figsize=(8, 6))
    colors = ["#2b6cb0", "#319795", "#d69e2e", "#805ad5"]
    
    for idx, (cls_name, color) in enumerate(zip(config.CLASS_NAMES, colors)):
        if len(np.unique(y_true_bin[:, idx])) >= 2:
            fpr, tpr, _ = roc_curve(y_true_bin[:, idx], test_probabilities[:, idx])
            roc_auc = auc(fpr, tpr)
            auc_records[config.DISPLAY_NAMES[cls_name]] = f"{roc_auc:.4f}"
            plt.plot(fpr, tpr, color=color, lw=2, label=f"{config.DISPLAY_NAMES[cls_name]} (AUC = {roc_auc:.3f})")
            
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Chance Line')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=11)
    plt.ylabel('True Positive Rate (Sensitivity)', fontsize=11)
    plt.title('One-vs-Rest ROC Curves per Category', fontsize=12, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(config.PLOTS_DIR / "roc_curves.png", dpi=160)
    plt.show()
    
    display(pd.DataFrame(list(auc_records.items()), columns=["Category", "One-vs-Rest AUC"]))
else:
    print("ROC / AUC curves pending actual model training.")


## 25. Single MRI Prediction Demo
Selecting a representative brain MRI from the test set and passing it through the exact inference pipeline used by the Streamlit application.


In [ ]:
# Select a sample test image
sample_test_path = sample_images[0][1]
sample_true_label = config.DISPLAY_NAMES[sample_images[0][0]]

print(f"Selected Test MRI Slice : {sample_test_path.name}")
print(f"True Category Label     : {sample_true_label}")

if config.MODEL_PATH.exists():
    pred_res = predict_with_model(sample_test_path, eval_model)
    print(f"\nModel Predicted Category : {pred_res['predicted_class']}")
    print(f"Model Confidence Score   : {pred_res['confidence'] * 100:.2f}%")
    print("\nFull Probability Distribution:")
    for cat_name, prob in pred_res["probabilities"].items():
        print(f"  {cat_name:18s}: {prob * 100:.2f}%")
else:
    print("Model prediction demo pending actual model training.")


## 26. Probability Distribution Bar Chart
Displaying the full softmax probability distribution for the analyzed MRI slice.


In [ ]:
if 'pred_res' in locals():
    prob_df = pd.DataFrame({
        "Category": list(pred_res["probabilities"].keys()),
        "Probability (%)": [v * 100.0 for v in pred_res["probabilities"].values()]
    })
    
    plt.figure(figsize=(8, 4))
    bars = plt.bar(prob_df["Category"], prob_df["Probability (%)"], color="#3182ce", edgecolor="black", width=0.5)
    plt.title(f"Class Probabilities: Predicted {pred_res['predicted_class']} ({pred_res['confidence']*100:.1f}%)", fontweight="bold")
    plt.xlabel("MRI Diagnostic Category")
    plt.ylabel("Probability (%)")
    plt.ylim(0, 100)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    for b in bars:
        h = b.get_height()
        plt.text(b.get_x() + b.get_width()/2.0, h + 1.5, f"{h:.1f}%", ha="center", va="bottom", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Probability distribution bar chart pending actual model training.")


## 27. Grad-CAM Explainability
**Gradient-weighted Class Activation Mapping (Grad-CAM)** visually explains where the neural network is looking.

By computing the gradient of the winning class score $y^c$ with respect to feature activation maps $A^k$ of the final convolutional layer:
$$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{i,j}^k}$$
$$L_{\text{Grad-CAM}}^c = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)$$

Grad-CAM indicates regions that influenced the decision. It is an explanatory tool, **not** a clinical tumor segmentation.


In [ ]:
if config.MODEL_PATH.exists():
    original_img, cam_heatmap, cam_overlay = create_heatmap_images(
        sample_test_path,
        eval_model,
        class_index=pred_res["class_index"]
    )
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(original_img)
    axes[0].set_title(f"Original MRI\n({sample_true_label})", fontsize=11, fontweight="bold")
    axes[0].axis("off")
    
    axes[1].imshow(cam_heatmap)
    axes[1].set_title("Grad-CAM Heatmap\n(Final DenseBlock Activation)", fontsize=11, fontweight="bold")
    axes[1].axis("off")
    
    axes[2].imshow(cam_overlay)
    axes[2].set_title(f"Heatmap Overlay\n(Predicted: {pred_res['predicted_class']})", fontsize=11, fontweight="bold")
    axes[2].axis("off")
    
    plt.suptitle("Model Visual Explainability via Grad-CAM", fontsize=13, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.show()
    print("⚠️ " + config.GRADCAM_DISCLAIMER)
else:
    print("Grad-CAM visualization pending actual model training.")


## 28. Final Results and Academic Conclusions
### Summary of the Lifecycle:
1. **Dataset Resolution**: Robust detection supporting Kaggle nested and custom paths.
2. **Exploratory Data Analysis**: Exact verification of class distributions and image integrity.
3. **Consistent Preprocessing**: Conversion to RGB, $224 \times 224$ resizing, float32 conversion, and DenseNet normalization across training and inference.
4. **Conservative Augmentation**: Applied strictly to training data to preserve anatomical validity.
5. **DenseNet121 Transfer Learning**: Pretrained ImageNet representations adapted in two distinct stages (feature extraction followed by selective fine-tuning).
6. **Rigorous Evaluation**: Held-out test evaluation avoiding data leakage, computing accuracy, macro precision, recall, F1, confusion matrix, and ROC/AUC.
7. **Explainability**: Grad-CAM heatmap tracing model saliency back to the final convolutional block.


## 29. Viva Voce Q&A Preparation Guide
Key questions and answers for academic examination:

1. **Why DenseNet121 instead of VGG or standard CNN?**
   - *Answer:* DenseNet introduces direct connections between all layers ($x_l = H_l([x_0, ..., x_{l-1}])$). This enables maximum feature reuse, alleviates vanishing gradients, and requires far fewer parameters (~7M vs ~138M for VGG16), achieving superior accuracy with faster training.

2. **Why use two training stages?**
   - *Answer:* Randomly initialized dense classification head weights produce large gradient updates initially. If the entire network were unfrozen immediately, these large gradients would destroy the valuable pretrained ImageNet representations. In Stage 1, the backbone is frozen. In Stage 2, upper layers are unfrozen with a 100x lower learning rate.

3. **Why keep BatchNormalization layers frozen during fine-tuning?**
   - *Answer:* BatchNormalization maintains running mean and variance estimates computed across ImageNet. Updating these statistics on small medical batches degrades pretrained representation quality.

4. **What does Grad-CAM compute?**
   - *Answer:* Grad-CAM computes the gradient of the target class score with respect to the feature maps of the last convolutional layer, pools the gradients to get importance weights $\alpha_k^c$, and computes a weighted ReLU combination to highlight positively contributing spatial regions.

5. **How is data leakage prevented?**
   - *Answer:* The dataset is partitioned into Training and Testing splits. The Training split is divided into 80% train and 20% validation. Augmentation is applied exclusively to training data. The Testing split is held out and only evaluated once after all training and checkpoint selection are finalized.
